In [37]:
import cv2
import importlib
import numpy as np
import construct
from pathlib import Path

importlib.reload(construct)
from construct import A, ConstrainedTileMix, FixedConstrainedTileMix
from analyze_water_cooling import load_video_memmap

layout = np.array([
    [0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0, 0],
], dtype=np.uint8)
layout_mask = np.repeat(np.repeat(layout * 255, 10, axis=0), 10, axis=1)

allowed_images = [
    {"image": cv2.imread("img1.jpg"), "layout_mask": layout_mask},
    {"image": cv2.imread("img2.jpg"), "layout_mask": layout_mask},
    {"image": cv2.imread("img3.jpg"), "layout_mask": layout_mask},
]
for i in range(3):
    img = allowed_images[i]['image']
    allowed_images[i]['image'] = img[5:-5, 5:-5]

transform = A.Compose([
    ConstrainedTileMix(tile_size=10, p=1.0),
])

image = cv2.imread("img2.jpg")
result = transform(image=image, tile_mix_metadata=allowed_images)
output = result["image"]

cv2.imwrite("output.jpg", output)



True

In [38]:
res_after_gaus = cv2.GaussianBlur(result['image'], (3, 3), 0)
res_after_gaus = np.hstack((result['image'], res_after_gaus))

In [39]:
cv2.imwrite("construct_after_gaus.jpg", res_after_gaus)

True

In [40]:
fig1_sl = [210 + 5, 70 + 5, 295 - 5, 155 -5]
def construct_video(
    layout: np.ndarray,
    bbox: list[float, float, float, float],
    input_file: str | Path,
    output_file: str | Path,
):
    x1, y1, x2, y2 = map(int, bbox)
    if x2 <= x1 or y2 <= y1:
        raise ValueError("bbox must be [x1, y1, x2, y2] with x2 > x1 and y2 > y1")
    video, fps = load_video_memmap(input_file, "data")
    height, width = video.shape[:2]
    fps = float(fps) if fps and fps > 0 else 30.0
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_file),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for {output_file}")
    
    transform = A.Compose([
        FixedConstrainedTileMix(tile_size=10, p=1.0, seed=42),
    ])

    crop_height = min(y2 - y1, layout.shape[0])
    crop_width = min(x2 - x1, layout.shape[1])
    if y1 < 0 or x1 < 0 or y1 + crop_height > height or x1 + crop_width > width:
        raise ValueError("bbox is outside the video")
    target_layout_mask = layout[:crop_height, :crop_width]

    try:
        for index in range(video.shape[2]):
            raw_frame = np.asarray(video[:, :, index], dtype=np.float32)
            finite = np.isfinite(raw_frame)
            if not finite.any():
                frame = np.zeros((height, width), dtype=np.uint8)
            else:
                low, high = np.nanpercentile(raw_frame, (2, 98))
                scale = max(float(high - low), 1e-6)
                frame = np.nan_to_num((raw_frame - low) / scale * 255, nan=0.0, posinf=255.0, neginf=0.0)
                frame = np.clip(frame, 0, 255).astype(np.uint8)
            crop = frame[y1:y1 + crop_height, x1:x1 + crop_width]
            allowed_img = [
                {'image': crop, "layout_mask": target_layout_mask}
            ]
            result = transform(image=crop, tile_mix_metadata=allowed_img)["image"]
            frame[y1:y1 + result.shape[0], x1:x1 + result.shape[1]] = result
            writer.write(cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR))
    finally:
        writer.release()
    return output_file
    
input_path = Path("data/water/water1.mat")
construct_video(
    layout=layout_mask,
    input_file=input_path,
    output_file="output.mp4",
    bbox=fig1_sl,
)

PosixPath('output.mp4')

In [41]:
video, fps = load_video_memmap(Path("data/water/water1.mat"), "data")
print(f"video shape: {video.shape}, fps: {fps or 30.0}")

video shape: (480, 640, 3000), fps: 30.0
